# GLAMMAR AI — FLUX.1 [schnell]

High-quality open image model.

1. Sign into Google. Choose **Runtime → Change runtime type → T4 GPU** (free GPUs are not guaranteed).
2. Choose **Runtime → Run all**, approve after reviewing, then use the controls at the bottom.

Loading takes several minutes and the model is large — a free T4 may run out of memory. Try 512px if that happens.

Weights download into this temporary session. Save results before it disconnects. This notebook runs interactively in Colab — it is not an API server. Free Colab policies: https://research.google.com/colaboratory/faq.html

In [ ]:
%pip -q install diffusers transformers accelerate safetensors sentencepiece protobuf ipywidgets
print("Dependencies installed. Continue below.")

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")
print("GPU:", torch.cuda.get_device_name(0))

import ipywidgets as w
from IPython.display import display, clear_output
from google.colab import output, userdata
from diffusers import FluxPipeline
output.enable_custom_widget_manager()

REPO = "black-forest-labs/FLUX.1-schnell"
try:
    TOKEN = userdata.get("HF_TOKEN")
except Exception:
    TOKEN = None

pipe = FluxPipeline.from_pretrained(REPO, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
print("Pipeline ready:", REPO)

prompt = w.Textarea(placeholder="Describe the image…", layout=w.Layout(width="100%", height="110px"))
steps = w.IntSlider(value=4, min=1, max=50, description="Steps")
guidance = w.FloatSlider(value=0, min=0, max=15, step=0.5, description="Guidance")
send = w.Button(description="Generate", button_style="primary", icon="play")
log = w.Output()

def on_send(_):
    send.disabled = True
    with log:
        clear_output(wait=True)
        try:
            text = prompt.value.strip()[:2000]
            if not text:
                raise ValueError("Describe the image first.")
            with torch.inference_mode():
                image = pipe(prompt=text, num_inference_steps=steps.value, guidance_scale=guidance.value, width=768, height=768).images[0]
            image.save("/content/glammar-image.png")
            display(image)
            print("Saved in the Colab Files panel: glammar-image.png")
        except Exception as e:
            print("Could not complete:", e)
        finally:
            send.disabled = False

send.on_click(on_send)
display(w.VBox([prompt, steps, guidance, send, log]))